In [ ]:
# Google Colab: mount Drive if available. When running from the repository, this is skipped.
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print("Not running in Colab; will look for backend/data locally.")


In [ ]:
# Example first-voyage request used to establish where the vessel becomes available.
origin_port = 'Newcastle'
destination_port = 'Paradip'
current_month = '2026-10-01'
quantity = 60  # metric tonnes (60,000 kg = 60 MT)
type_of_cargo = 'coal'

# Fixed planning assumptions, in USD. These are assumptions, not market quotes.
reposition_cost_per_nm = 1.50
idle_cost_per_day = 5000.00


This corrected copy keeps the forecasting demonstration and replaces the old next-order section. The next-order calculation now starts from the vessel's actual post-voyage position, checks real open orders and schedule/capacity constraints, forecasts the next route's rate, multiplies USD/tonne by order quantity, then adds repositioning and idle costs. It reports measured holdout error; no 99% accuracy claim is assumed.

stage 1: constriaining

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from pathlib import Path

# Prefer the original Google Drive folder in Colab; otherwise use this repository's backend/data folder.
possible_data_dirs = [
    Path('/content/drive/My Drive/IAC 2026'),
    Path('backend/data'),
    Path('../backend/data'),
]
DATA_DIR = next((p for p in possible_data_dirs if (p / 'vessels.csv').exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError("Could not find vessels.csv. Set DATA_DIR to the folder containing the six CSV files.")
print("Using datasets from:", DATA_DIR.resolve())

df1 = pd.read_csv(DATA_DIR / 'vessels.csv')
df2 = pd.read_csv(DATA_DIR / 'ports.csv')
df3 = pd.read_csv(DATA_DIR / 'freight_rates.csv')
df4 = pd.read_csv(DATA_DIR / 'market_indicators.csv')
df5 = pd.read_csv(DATA_DIR / 'route_distances.csv')
df6 = pd.read_csv(DATA_DIR / 'cargo_orders.csv')


In [ ]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 16 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   vessel_id                         200 non-null    object 
 1   vessel_name                       200 non-null    object 
 2   vessel_type                       200 non-null    object 
 3   deadweight_tonnage_dwt            200 non-null    int64  
 4   usable_cargo_capacity_mt          200 non-null    int64  
 5   length_overall_m                  200 non-null    float64
 6   beam_m                            200 non-null    float64
 7   loaded_draft_m                    200 non-null    float64
 8   speed_knots                       200 non-null    float64
 9   age_years                         200 non-null    int64  
 10  cargo_compatibility               200 non-null    object 
 11  daily_idle_cost_usd               200 non-null    float64
 12  repositi

In [ ]:
df1

,vessel_id,vessel_name,vessel_type,deadweight_tonnage_dwt,usable_cargo_capacity_mt,length_overall_m,beam_m,loaded_draft_m,speed_knots,age_years,cargo_compatibility,daily_idle_cost_usd,repositioning_cost_per_nm_usd,assumed_monthly_charter_cost_usd,initial_origin_port_id,initial_available_date
0,VSL_001,MV Coromandel Star 001,Supramax,54874,50841,193.9,32.1,12.15,14.1,17,coal|iron_ore|wheat|fertilizer|limestone,10613.06,32.09,419950.57,PRT_002,2026-01-06
1,VSL_002,MV Southern Star 002,Handysize,27110,25561,169.8,26.8,9.21,14.1,19,coal|iron_ore|wheat|fertilizer|limestone,6672.10,21.76,244739.62,PRT_007,2026-01-18
2,VSL_003,MV Coromandel Horizon 003,Handysize,34459,32842,181.5,28.1,9.96,13.3,16,coal|wheat|fertilizer|limestone,8411.48,23.28,295003.26,PRT_001,2026-01-16
3,VSL_004,MV Coromandel Crest 004,Panamax,82406,78161,231.7,35.5,14.73,14.0,1,coal|iron_ore,18090.97,47.45,650889.00,PRT_003,2026-01-16
4,VSL_005,MV Indo Voyager 005,Panamax,78616,72344,230.2,34.6,14.45,13.1,1,coal|iron_ore,17215.03,43.48,590386.58,PRT_020,2026-01-08
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,VSL_196,MV Coromandel Courage 196,Supramax,62873,57877,202.1,32.6,12.82,13.5,12,coal|iron_ore|wheat|fertilizer|limestone,13746.93,37.50,494353.63,PRT_014,2026-01-05
196,VSL_197,MV Indo Voyager 197,Handysize,35972,33556,186.5,28.9,10.22,13.6,4,coal|wheat|fertilizer|limestone,8410.29,25.08,311870.13,PRT_015,2026-01-14
197,VSL_198,MV Indo Voyager 198,Supramax,61423,58170,201.0,32.4,13.00,13.8,18,coal|iron_ore|limestone,13152.62,36.15,477225.85,PRT_017,2026-01-12
198,VSL_199,MV Kalinga Voyager 199,Capesize,146426,137894,279.8,44.0,16.51,13.0,7,coal|iron_ore,20868.73,56.92,769690.51,PRT_012,2026-01-19


In [ ]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 13 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   port_id                             20 non-null     object 
 1   port_name                           20 non-null     object 
 2   country                             20 non-null     object 
 3   port_role                           20 non-null     object 
 4   latitude                            20 non-null     float64
 5   longitude                           20 non-null     float64
 6   max_permissible_draft_m             20 non-null     float64
 7   max_vessel_loa_m                    20 non-null     int64  
 8   max_vessel_beam_m                   20 non-null     int64  
 9   max_berth_dwt                       20 non-null     int64  
 10  cargo_handling_rate_tonnes_per_day  20 non-null     int64  
 11  supported_cargo_types               20 non-null

In [ ]:
df2

,port_id,port_name,country,port_role,latitude,longitude,max_permissible_draft_m,max_vessel_loa_m,max_vessel_beam_m,max_berth_dwt,cargo_handling_rate_tonnes_per_day,supported_cargo_types,port_congestion_index
0,PRT_001,Newcastle,Australia,loading,-32.9283,151.7817,16.5,300,50,210000,45000,coal,0.25
1,PRT_002,Port Hedland,Australia,loading,-20.3106,118.5753,18.5,330,55,250000,65000,iron_ore,0.30
2,PRT_003,Gladstone,Australia,loading,-23.8427,151.2525,16.0,295,48,190000,40000,coal|wheat,0.20
3,PRT_004,Samarinda,Indonesia,loading,-0.5022,117.1536,13.5,240,40,95000,28000,coal,0.45
4,PRT_005,Banjarmasin,Indonesia,loading,-3.3194,114.5908,12.8,229,38,85000,25000,coal,0.50
5,PRT_006,Tanjung Bara,Indonesia,loading,0.5283,117.6514,17.2,300,50,200000,48000,coal,0.22
6,PRT_007,Richards Bay,South Africa,loading,-28.8000,32.0833,17.5,310,50,210000,42000,coal|iron_ore,0.38
7,PRT_008,Saldanha Bay,South Africa,loading,-33.0117,17.9447,19.0,330,55,240000,55000,iron_ore,0.28
8,PRT_009,Maputo,Mozambique,loading,-25.9692,32.5732,14.2,255,43,110000,26000,coal|iron_ore,0.42
9,PRT_010,Beira,Mozambique,loading,-19.8333,34.8500,11.8,205,33,65000,18000,coal|wheat|fertilizer,0.48


In [ ]:
loa = df2[df2['port_name'] == destination_port]['max_vessel_loa_m'].values[0]
draft = df2[df2['port_name'] == destination_port]['max_permissible_draft_m'].values[0]
beam = df2[df2['port_name'] == destination_port]['max_vessel_beam_m'].values[0]
print('loa: ', loa)
print('draft: ', draft)
print('beam: ', beam)

loa:  315
draft:  17.8
beam:  52


In [ ]:
const_df = df1[(df1['length_overall_m'] < loa) & (df1['loaded_draft_m'] < draft) & (df1['beam_m'] < beam) & (df1['deadweight_tonnage_dwt'] > quantity)]
const_df

,vessel_id,vessel_name,vessel_type,deadweight_tonnage_dwt,usable_cargo_capacity_mt,length_overall_m,beam_m,loaded_draft_m,speed_knots,age_years,cargo_compatibility,daily_idle_cost_usd,repositioning_cost_per_nm_usd,assumed_monthly_charter_cost_usd,initial_origin_port_id,initial_available_date
3,VSL_004,MV Coromandel Crest 004,Panamax,82406,78161,231.7,35.5,14.73,14.0,1,coal|iron_ore,18090.97,47.45,650889.00,PRT_003,2026-01-16
4,VSL_005,MV Indo Voyager 005,Panamax,78616,72344,230.2,34.6,14.45,13.1,1,coal|iron_ore,17215.03,43.48,590386.58,PRT_020,2026-01-08
7,VSL_008,MV Eastern Crest 008,Capesize,174544,167933,292.6,47.6,17.32,13.6,2,coal|iron_ore,23768.52,65.34,895290.89,PRT_017,2026-01-08
8,VSL_009,MV Southern Voyager 009,Capesize,186198,174314,297.4,47.7,17.67,13.5,7,coal|iron_ore,25399.44,67.35,949826.22,PRT_008,2026-01-14
11,VSL_012,MV Indo Pioneer 012,Capesize,171304,160089,289.5,47.0,17.04,12.2,21,coal|iron_ore|limestone,23612.28,66.28,887631.04,PRT_001,2026-01-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
188,VSL_189,MV Indo Empress 189,Supramax,61186,56291,201.8,31.4,12.77,13.5,3,coal|iron_ore|limestone,12755.04,36.22,478424.44,PRT_005,2026-01-13
191,VSL_192,MV Pacific Crest 192,Capesize,186679,173190,296.2,48.1,17.57,12.3,6,coal|iron_ore,25262.90,67.97,955508.53,PRT_002,2026-01-09
195,VSL_196,MV Coromandel Courage 196,Supramax,62873,57877,202.1,32.6,12.82,13.5,12,coal|iron_ore|wheat|fertilizer|limestone,13746.93,37.50,494353.63,PRT_014,2026-01-05
197,VSL_198,MV Indo Voyager 198,Supramax,61423,58170,201.0,32.4,13.00,13.8,18,coal|iron_ore|limestone,13152.62,36.15,477225.85,PRT_017,2026-01-12


In [ ]:
const_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 78 entries, 3 to 198
Data columns (total 16 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   vessel_id                         78 non-null     object 
 1   vessel_name                       78 non-null     object 
 2   vessel_type                       78 non-null     object 
 3   deadweight_tonnage_dwt            78 non-null     int64  
 4   usable_cargo_capacity_mt          78 non-null     int64  
 5   length_overall_m                  78 non-null     float64
 6   beam_m                            78 non-null     float64
 7   loaded_draft_m                    78 non-null     float64
 8   speed_knots                       78 non-null     float64
 9   age_years                         78 non-null     int64  
 10  cargo_compatibility               78 non-null     object 
 11  daily_idle_cost_usd               78 non-null     float64
 12  repositioning_

After constraining we should predict corresponding charter rates for these vessels using ml model. before that we should train it in order to get predict

Step 1: dataframe creation

In [ ]:
df3.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18480 entries, 0 to 18479
Data columns (total 14 columns):
 #   Column                                  Non-Null Count  Dtype  
---  ------                                  --------------  -----  
 0   rate_id                                 18480 non-null  object 
 1   date                                    18480 non-null  object 
 2   route_id                                18480 non-null  object 
 3   origin_port_id                          18480 non-null  object 
 4   destination_port_id                     18480 non-null  object 
 5   vessel_type                             18480 non-null  object 
 6   distance_nm                             18480 non-null  float64
 7   freight_rate_usd_per_tonne              18480 non-null  float64
 8   fuel_price_usd_per_tonne                18480 non-null  float64
 9   relevant_commodity_price_usd_per_tonne  18480 non-null  float64
 10  origin_congestion_index                 18480 non-null  fl

In [ ]:
df3

,rate_id,date,route_id,origin_port_id,destination_port_id,vessel_type,distance_nm,freight_rate_usd_per_tonne,fuel_price_usd_per_tonne,relevant_commodity_price_usd_per_tonne,origin_congestion_index,destination_congestion_index,shipping_market_index,seasonal_demand_index
0,FRT_000001,2016-01-01,RTE_001,PRT_001,PRT_014,Handysize,6480.4,41.59,462.45,79.36,0.2056,0.3165,1363.53,0.8788
1,FRT_000002,2016-02-01,RTE_001,PRT_001,PRT_014,Handysize,6480.4,46.68,495.89,80.02,0.2010,0.3957,1394.12,0.9300
2,FRT_000003,2016-03-01,RTE_001,PRT_001,PRT_014,Handysize,6480.4,52.98,463.22,88.89,0.2058,0.4663,1531.70,1.0000
3,FRT_000004,2016-04-01,RTE_001,PRT_001,PRT_014,Handysize,6480.4,57.76,457.52,105.26,0.1906,0.3964,1562.83,1.0700
4,FRT_000005,2016-05-01,RTE_001,PRT_001,PRT_014,Handysize,6480.4,70.76,534.92,109.63,0.2633,0.5050,1860.69,1.1212
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18475,FRT_018476,2026-08-01,RTE_108,PRT_019,PRT_018,Capesize,285.9,6.96,440.45,70.76,0.3587,0.6052,1417.85,1.0700
18476,FRT_018477,2026-09-01,RTE_108,PRT_019,PRT_018,Capesize,285.9,5.60,512.47,79.51,0.3111,0.5777,1373.75,1.0000
18477,FRT_018478,2026-10-01,RTE_108,PRT_019,PRT_018,Capesize,285.9,4.95,503.48,66.01,0.2565,0.5143,1436.25,0.9300
18478,FRT_018479,2026-11-01,RTE_108,PRT_019,PRT_018,Capesize,285.9,4.52,524.54,67.59,0.3715,0.5891,1532.99,0.8788


In [ ]:
df4.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 132 entries, 0 to 131
Data columns (total 11 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   date                              132 non-null    object 
 1   coal_price_usd_per_tonne          132 non-null    float64
 2   iron_ore_price_usd_per_tonne      132 non-null    float64
 3   wheat_price_usd_per_tonne         132 non-null    float64
 4   bunker_fuel_price_usd_per_tonne   132 non-null    float64
 5   brent_crude_price_usd_per_barrel  132 non-null    float64
 6   usd_inr_exchange_rate             132 non-null    float64
 7   shipping_market_index             132 non-null    float64
 8   global_bulk_cargo_demand_index    132 non-null    float64
 9   port_congestion_index             132 non-null    float64
 10  global_shipping_capacity_index    132 non-null    float64
dtypes: float64(10), object(1)
memory usage: 11.5+ KB


In [ ]:
df4

,date,coal_price_usd_per_tonne,iron_ore_price_usd_per_tonne,wheat_price_usd_per_tonne,bunker_fuel_price_usd_per_tonne,brent_crude_price_usd_per_barrel,usd_inr_exchange_rate,shipping_market_index,global_bulk_cargo_demand_index,port_congestion_index,global_shipping_capacity_index
0,2016-01-01,79.36,95.50,226.25,462.45,62.99,66.64,1363.53,95.92,0.2693,100.85
1,2016-02-01,80.02,90.06,213.66,495.89,65.88,66.88,1394.12,98.95,0.2635,99.85
2,2016-03-01,88.89,110.88,221.32,463.22,68.82,67.07,1531.70,105.50,0.3021,99.05
3,2016-04-01,105.26,112.52,194.38,457.52,68.57,67.17,1562.83,105.19,0.3179,101.90
4,2016-05-01,109.63,116.96,215.66,534.92,72.46,67.53,1860.69,115.32,0.3964,99.40
...,...,...,...,...,...,...,...,...,...,...,...
127,2026-08-01,70.76,75.31,210.52,440.45,66.08,82.33,1417.85,117.51,0.3443,117.94
128,2026-09-01,79.51,82.84,205.81,512.47,72.28,82.30,1373.75,115.17,0.2994,117.37
129,2026-10-01,66.01,73.57,224.75,503.48,75.06,82.32,1436.25,115.65,0.2720,117.80
130,2026-11-01,67.59,81.03,212.16,524.54,74.72,82.33,1532.99,115.91,0.3193,118.09


In [ ]:
# Merge df3 and df4 on 'observation_month' index
df_merged = df3.merge(df4, on = 'date', how='inner').set_index('date')

# Display the merged dataset
display(df_merged.head())

,rate_id,route_id,origin_port_id,destination_port_id,vessel_type,distance_nm,freight_rate_usd_per_tonne,fuel_price_usd_per_tonne,relevant_commodity_price_usd_per_tonne,origin_congestion_index,...,coal_price_usd_per_tonne,iron_ore_price_usd_per_tonne,wheat_price_usd_per_tonne,bunker_fuel_price_usd_per_tonne,brent_crude_price_usd_per_barrel,usd_inr_exchange_rate,shipping_market_index_y,global_bulk_cargo_demand_index,port_congestion_index,global_shipping_capacity_index
date,,,,,,,,,,,,,,,,,,,,,
2016-01-01,FRT_000001,RTE_001,PRT_001,PRT_014,Handysize,6480.4,41.59,462.45,79.36,0.2056,...,79.36,95.50,226.25,462.45,62.99,66.64,1363.53,95.92,0.2693,100.85
2016-02-01,FRT_000002,RTE_001,PRT_001,PRT_014,Handysize,6480.4,46.68,495.89,80.02,0.2010,...,80.02,90.06,213.66,495.89,65.88,66.88,1394.12,98.95,0.2635,99.85
2016-03-01,FRT_000003,RTE_001,PRT_001,PRT_014,Handysize,6480.4,52.98,463.22,88.89,0.2058,...,88.89,110.88,221.32,463.22,68.82,67.07,1531.70,105.50,0.3021,99.05
2016-04-01,FRT_000004,RTE_001,PRT_001,PRT_014,Handysize,6480.4,57.76,457.52,105.26,0.1906,...,105.26,112.52,194.38,457.52,68.57,67.17,1562.83,105.19,0.3179,101.90
2016-05-01,FRT_000005,RTE_001,PRT_001,PRT_014,Handysize,6480.4,70.76,534.92,109.63,0.2633,...,109.63,116.96,215.66,534.92,72.46,67.53,1860.69,115.32,0.3964,99.40


In [ ]:
df_merged.info()

<class 'pandas.core.frame.DataFrame'>
Index: 18480 entries, 2016-01-01 to 2026-12-01
Data columns (total 23 columns):
 #   Column                                  Non-Null Count  Dtype  
---  ------                                  --------------  -----  
 0   rate_id                                 18480 non-null  object 
 1   route_id                                18480 non-null  object 
 2   origin_port_id                          18480 non-null  object 
 3   destination_port_id                     18480 non-null  object 
 4   vessel_type                             18480 non-null  object 
 5   distance_nm                             18480 non-null  float64
 6   freight_rate_usd_per_tonne              18480 non-null  float64
 7   fuel_price_usd_per_tonne                18480 non-null  float64
 8   relevant_commodity_price_usd_per_tonne  18480 non-null  float64
 9   origin_congestion_index                 18480 non-null  float64
 10  destination_congestion_index            18480 non

Step 2: Training starts

Note: this step does not happen every time user enters input

In [ ]:
%pip install -q xgboost

In [ ]:

# ============================================================
# CELL 2: TRAIN XGBOOST + FOUR-MONTH FREIGHT-RATE FORECAST
# Input: df_merged, with dates stored in its row index
#
# OUTPUT 1: MAE, RMSE, R² for forecast horizons 1, 2, 3, 4
# OUTPUT 2: Historical input date -> next 4 months
#           Actual vs predicted table + graph
# OUTPUT 3: Synthetic future input date outside df_merged
#           -> next 4 months, with no graph
# ============================================================

# Uncomment if XGBoost is not installed in your Colab:
# %pip install -q xgboost

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

# ------------------------------------------------------------
# 0. SETTINGS YOU CAN CHANGE
# ------------------------------------------------------------

TARGET = "freight_rate_usd_per_tonne"

# Leave as None to automatically select a historical date
# somewhere inside the dataset, with 4 subsequent actual months.
HISTORICAL_INPUT_DATE = None

# Leave as None to automatically select a route/vessel series.
# Or specify existing values, for example:
# SELECTED_ROUTE = "route_001"
# SELECTED_VESSEL_TYPE = "Panamax"
SELECTED_ROUTE = None
SELECTED_VESSEL_TYPE = None

# Synthetic input date must be beyond the dataset's last month.
# Example: "2030-01-01"
SYNTHETIC_INPUT_DATE = "2030-01-01"

FORECAST_HORIZONS = [1, 2, 3, 4]
RANDOM_STATE = 42

# ------------------------------------------------------------
# 1. PREPARE df_merged
# ------------------------------------------------------------

if "df_merged" not in globals():
    raise NameError("df_merged is not defined. Run your data-loading cell first.")

if TARGET not in df_merged.columns:
    raise ValueError(f"Target column '{TARGET}' was not found.")

raw = df_merged.copy()

# Dates are in the index, as shown in your DataFrame output.
raw.index = pd.to_datetime(raw.index, errors="coerce")
raw = raw.loc[raw.index.notna()].copy()
raw.index.name = "date"
raw = raw.reset_index()

# Normalize each record to its calendar month.
raw["date"] = pd.to_datetime(raw["date"]).dt.to_period("M").dt.to_timestamp()

# Remove identifier columns that should not be model features.
raw = raw.drop(columns=["rate_id"], errors="ignore")

# Remove rows without a target.
raw[TARGET] = pd.to_numeric(raw[TARGET], errors="coerce")
raw = raw.dropna(subset=[TARGET]).copy()

# Route/vessel/port identifiers define a series.
# If any are missing, use a consistent placeholder.
GROUP_COLS = [
    "route_id",
    "origin_port_id",
    "destination_port_id",
    "vessel_type"
]

for col in GROUP_COLS:
    if col not in raw.columns:
        raw[col] = "Unknown"
    raw[col] = raw[col].fillna("Unknown").astype(str)

# Numeric predictors, excluding the target.
# All available numeric market, cost, congestion and distance columns
# are aggregated by month for each route/port/vessel combination.
NUMERIC_COLS = [
    c for c in raw.select_dtypes(include=[np.number]).columns
    if c != TARGET
]

# Keep one monthly observation for each route/ports/vessel series.
AGG = {c: "mean" for c in NUMERIC_COLS}
AGG[TARGET] = "mean"

monthly = (
    raw.groupby(["date"] + GROUP_COLS, as_index=False, dropna=False)
       .agg(AGG)
       .sort_values(["route_id", "origin_port_id",
                     "destination_port_id", "vessel_type", "date"])
       .reset_index(drop=True)
)

if monthly.empty:
    raise ValueError("No usable monthly records remain after preparation.")

last_historical_date = monthly["date"].max()
first_historical_date = monthly["date"].min()

print("=" * 65)
print("DATA PREPARATION")
print("=" * 65)
print(f"Historical data: {first_historical_date:%Y-%m} to "
      f"{last_historical_date:%Y-%m}")
print(f"Monthly route/port/vessel records: {len(monthly):,}")
print(f"Distinct routes: {monthly['route_id'].nunique()}")
print(f"Distinct vessel types: {monthly['vessel_type'].nunique()}")
print(f"Target: {TARGET}")

# ------------------------------------------------------------
# 2. CREATE LAG FEATURES AND FOUR FUTURE TARGETS
# ------------------------------------------------------------

SERIES_COLS = GROUP_COLS
grouped = monthly.groupby(SERIES_COLS, sort=False, dropna=False)

# Previous freight rates are known at the input/origin date.
for lag in [1, 2, 3, 6, 12]:
    monthly[f"rate_lag_{lag}"] = grouped[TARGET].shift(lag)

# Rolling statistics use only rates strictly before the origin date.
grouped = monthly.groupby(SERIES_COLS, sort=False, dropna=False)

monthly["rate_rolling_mean_3"] = grouped[TARGET].transform(
    lambda s: s.shift(1).rolling(3, min_periods=1).mean()
)
monthly["rate_rolling_mean_6"] = grouped[TARGET].transform(
    lambda s: s.shift(1).rolling(6, min_periods=1).mean()
)
monthly["rate_rolling_mean_12"] = grouped[TARGET].transform(
    lambda s: s.shift(1).rolling(12, min_periods=1).mean()
)

# Each target is the freight rate 1, 2, 3 or 4 observations
# ahead within the same route/port/vessel series.
for h in FORECAST_HORIZONS:
    monthly[f"target_h{h}"] = (
        monthly.groupby(SERIES_COLS, sort=False, dropna=False)[TARGET]
        .shift(-h)
    )
    monthly[f"target_date_h{h}"] = (
        monthly.groupby(SERIES_COLS, sort=False, dropna=False)["date"]
        .shift(-h)
    )

# Use the forecast month for seasonality/time features.
# This makes January forecasts different from July forecasts.
BASE_DATE = first_historical_date

def add_target_calendar_features(frame, target_dates):
    out = frame.copy()
    target_dates = pd.to_datetime(target_dates)

    month_number = target_dates.dt.month if isinstance(
        target_dates, pd.Series
    ) else target_dates.month

    out["target_month_sin"] = np.sin(2 * np.pi * month_number / 12)
    out["target_month_cos"] = np.cos(2 * np.pi * month_number / 12)

    out["target_time_index"] = (
        (target_dates.dt.year - BASE_DATE.year) * 12
        + (target_dates.dt.month - BASE_DATE.month)
        if isinstance(target_dates, pd.Series)
        else (target_dates.year - BASE_DATE.year) * 12
             + (target_dates.month - BASE_DATE.month)
    )
    return out

# Model features: current known market/vessel/route data plus rate history.
LAG_COLS = [
    "rate_lag_1", "rate_lag_2", "rate_lag_3",
    "rate_lag_6", "rate_lag_12",
    "rate_rolling_mean_3", "rate_rolling_mean_6",
    "rate_rolling_mean_12"
]

CATEGORICAL_FEATURES = GROUP_COLS.copy()

EXCLUDED = {
    "date", TARGET, "rate_id",
    *[f"target_h{h}" for h in FORECAST_HORIZONS],
    *[f"target_date_h{h}" for h in FORECAST_HORIZONS]
}

BASE_FEATURES = (
    CATEGORICAL_FEATURES
    + NUMERIC_COLS
    + LAG_COLS
)

# Keep features unique and only those that exist.
BASE_FEATURES = list(dict.fromkeys([
    c for c in BASE_FEATURES
    if c in monthly.columns and c not in EXCLUDED
]))

# Add the known target-month calendar features.
FEATURES = BASE_FEATURES + [
    "target_month_sin",
    "target_month_cos",
    "target_time_index"
]

# ------------------------------------------------------------
# 3. CHRONOLOGICAL TRAIN/TEST SPLIT
# ------------------------------------------------------------

all_dates = np.array(sorted(monthly["date"].unique()))

if len(all_dates) < 18:
    raise ValueError(
        "Not enough distinct months for a reliable four-month forecast. "
        "At least 18 distinct months are recommended."
    )

# Keep the latest ~20% of origin dates for evaluation.
split_position = int(len(all_dates) * 0.80)
split_position = min(max(split_position, 1), len(all_dates) - 5)
split_date = pd.Timestamp(all_dates[split_position])

print(f"\nChronological evaluation split starts: {split_date:%Y-%m}")

# ------------------------------------------------------------
# 4. TRAIN FOUR DIRECT XGBOOST MODELS
# ------------------------------------------------------------

models = {}
evaluation_rows = []
test_predictions = {}

for h in FORECAST_HORIZONS:

    target_col = f"target_h{h}"
    target_date_col = f"target_date_h{h}"

    # Build calendar features for the relevant forecast month.
    frame = add_target_calendar_features(
        monthly[["date"] + BASE_FEATURES + [target_col, target_date_col]]
        .copy(),
        monthly[target_date_col]
    )

    frame = frame.dropna(subset=[target_col, target_date_col]).copy()
    frame[target_date_col] = pd.to_datetime(frame[target_date_col])

    # Training targets must be dated before the test-origin split.
    train_mask = frame[target_date_col] < split_date

    # Test predictions originate at/after the split date.
    # The target must still be within the known historical period.
    test_mask = (
        (frame["date"] >= split_date)
        & (frame[target_date_col] <= last_historical_date)
    )

    train_frame = frame.loc[train_mask].copy()
    test_frame = frame.loc[test_mask].copy()

    if len(train_frame) < 50:
        raise ValueError(
            f"Not enough training rows for horizon {h}. "
            "Check your dataset and route/vessel coverage."
        )

    if len(test_frame) == 0:
        raise ValueError(
            f"No test rows for horizon {h}. Try a different split or "
            "check whether the data contains enough future observations."
        )

    cat_cols = [c for c in CATEGORICAL_FEATURES if c in FEATURES]
    num_cols = [c for c in FEATURES if c not in cat_cols]

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "categorical",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore"))
                ]),
                cat_cols
            ),
            (
                "numeric",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="median"))
                ]),
                num_cols
            )
        ],
        remainder="drop"
    )

    model = Pipeline([
        ("preprocessor", preprocessor),
        ("xgb", XGBRegressor(
            n_estimators=350,
            max_depth=5,
            learning_rate=0.04,
            subsample=0.85,
            colsample_bytree=0.85,
            objective="reg:squarederror",
            random_state=RANDOM_STATE,
            n_jobs=-1
        ))
    ])

    model.fit(train_frame[FEATURES], train_frame[target_col])

    pred = model.predict(test_frame[FEATURES])
    actual = test_frame[target_col].to_numpy()

    models[h] = model
    test_predictions[h] = (
        test_frame[["date", target_date_col] + GROUP_COLS]
        .assign(actual=actual, predicted=pred)
        .reset_index(drop=True)
    )

    abs_error = np.abs(actual - pred)
    nonzero_actual = np.abs(actual) > 1e-9
    evaluation_rows.append({
        "Forecast horizon": f"Month +{h}",
        "Training rows": len(train_frame),
        "Test rows": len(test_frame),
        "MAE (USD/tonne)": mean_absolute_error(actual, pred),
        "RMSE (USD/tonne)": np.sqrt(mean_squared_error(actual, pred)),
        "R² (not accuracy)": r2_score(actual, pred),
        "WAPE (%)": 100 * abs_error.sum() / max(np.abs(actual).sum(), 1e-9),
        "Within 10% (%)": 100 * np.mean(abs_error[nonzero_actual] <= 0.10 * np.abs(actual[nonzero_actual])) if nonzero_actual.any() else np.nan
    })

accuracy_df = pd.DataFrame(evaluation_rows)

print("\n" + "=" * 65)
print("OUTPUT 1 — TEMPORAL HOLDOUT ERROR (NO GRAPH)")
print("=" * 65)
display(
    accuracy_df.style.format({
        "MAE (USD/tonne)": "{:.3f}",
        "RMSE (USD/tonne)": "{:.3f}",
        "R² (not accuracy)": "{:.4f}",
        "WAPE (%)": "{:.2f}",
        "Within 10% (%)": "{:.1f}"
    })
)

# ------------------------------------------------------------
# 5. REFIT MODELS USING ALL AVAILABLE LABELED HISTORY
#    These are the models used for the sample forecasts.
# ------------------------------------------------------------

final_models = {}

for h in FORECAST_HORIZONS:
    target_col = f"target_h{h}"
    target_date_col = f"target_date_h{h}"

    frame = add_target_calendar_features(
        monthly[["date"] + BASE_FEATURES + [target_col, target_date_col]]
        .copy(),
        monthly[target_date_col]
    )
    frame = frame.dropna(subset=[target_col, target_date_col]).copy()

    cat_cols = [c for c in CATEGORICAL_FEATURES if c in FEATURES]
    num_cols = [c for c in FEATURES if c not in cat_cols]

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "categorical",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore"))
                ]),
                cat_cols
            ),
            (
                "numeric",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="median"))
                ]),
                num_cols
            )
        ],
        remainder="drop"
    )

    final_model = Pipeline([
        ("preprocessor", preprocessor),
        ("xgb", XGBRegressor(
            n_estimators=350,
            max_depth=5,
            learning_rate=0.04,
            subsample=0.85,
            colsample_bytree=0.85,
            objective="reg:squarederror",
            random_state=RANDOM_STATE,
            n_jobs=-1
        ))
    ])

    final_model.fit(frame[FEATURES], frame[target_col])
    final_models[h] = final_model

# ------------------------------------------------------------
# 6. SELECT ONE ROUTE/VESSEL SERIES
# ------------------------------------------------------------

series_counts = (
    monthly.groupby(GROUP_COLS, dropna=False)
    .size()
    .reset_index(name="n_months")
    .query("n_months >= 16")
    .sort_values("n_months", ascending=False)
)

if series_counts.empty:
    raise ValueError(
        "No route/port/vessel series has enough observations. "
        "Check your grouping columns and dataset."
    )

if SELECTED_ROUTE is not None:
    series_counts = series_counts[
        series_counts["route_id"] == str(SELECTED_ROUTE)
    ]

if SELECTED_VESSEL_TYPE is not None:
    series_counts = series_counts[
        series_counts["vessel_type"] == str(SELECTED_VESSEL_TYPE)
    ]

if series_counts.empty:
    raise ValueError(
        "The selected route/vessel combination was not found with enough "
        "historical records. Check SELECTED_ROUTE and SELECTED_VESSEL_TYPE."
    )

chosen = series_counts.iloc[0]
chosen_key = {c: chosen[c] for c in GROUP_COLS}

selected_series = monthly.copy()
for c in GROUP_COLS:
    selected_series = selected_series[selected_series[c] == chosen_key[c]]

selected_series = selected_series.sort_values("date").reset_index(drop=True)

print("\nSelected sample series:")
print(chosen_key)

# ------------------------------------------------------------
# 7. HELPER: BUILD FEATURES FOR A FOUR-MONTH FORECAST
# ------------------------------------------------------------

def build_forecast_features(history, origin_row, forecast_month):
    """
    Use features available at origin_row's date.
    Lag features are calculated from that route/vessel's actual
    freight-rate history up to and including the origin date.
    """
    row = origin_row.copy()

    # Rate lags: the most recent observed rate is lag 1 relative
    # to a forecast origin after that observation.
    prior_rates = history.loc[
        history["date"] <= origin_row["date"], TARGET
    ].astype(float).tolist()

    for lag in [1, 2, 3, 6, 12]:
        row[f"rate_lag_{lag}"] = (
            prior_rates[-lag] if len(prior_rates) >= lag else np.nan
        )

    for window in [3, 6, 12]:
        values = prior_rates[-window:]
        row[f"rate_rolling_mean_{window}"] = (
            float(np.mean(values)) if len(values) else np.nan
        )

    month_num = forecast_month.month
    row["target_month_sin"] = np.sin(2 * np.pi * month_num / 12)
    row["target_month_cos"] = np.cos(2 * np.pi * month_num / 12)
    row["target_time_index"] = (
        (forecast_month.year - BASE_DATE.year) * 12
        + (forecast_month.month - BASE_DATE.month)
    )

    return pd.DataFrame([{f: row.get(f, np.nan) for f in FEATURES}])

def predict_four_months(history, origin_row, first_forecast_month, model_set):
    outputs = []

    for h in FORECAST_HORIZONS:
        forecast_month = (
            first_forecast_month + pd.DateOffset(months=h - 1)
        )
        x = build_forecast_features(history, origin_row, forecast_month)
        predicted_rate = max(0.0, float(model_set[h].predict(x[FEATURES])[0]))

        outputs.append({
            "Forecast month": forecast_month,
            "Predicted freight rate (USD/tonne)": predicted_rate
        })

    return pd.DataFrame(outputs)

# ------------------------------------------------------------
# 8. OUTPUT 2 — HISTORICAL INPUT + NEXT 4 MONTHS
#    Choose a date well inside the dataset, NOT the last month.
#    Compare predictions with actual values for all 4 months.
# ------------------------------------------------------------

if HISTORICAL_INPUT_DATE is None:
    # Use an origin date in the chronological holdout, after the model's
    # training cutoff. Require the next four calendar months to exist.
    possible_origins = selected_series[
        (selected_series["date"] >= split_date)
        & (selected_series["date"] <= last_historical_date - pd.DateOffset(months=4))
    ]
    historical_origin = None
    for _, candidate in possible_origins.iterrows():
        expected_months = [candidate["date"] + pd.DateOffset(months=h) for h in FORECAST_HORIZONS]
        observed_months = selected_series.loc[
            selected_series["date"] > candidate["date"], "date"
        ].head(4).tolist()
        if len(observed_months) == 4 and all(
            pd.Timestamp(actual).to_period("M") == pd.Timestamp(expected).to_period("M")
            for actual, expected in zip(observed_months, expected_months)
        ):
            historical_origin = candidate
            break
    if historical_origin is None:
        raise ValueError("No four-consecutive-month sample exists in the chronological holdout for this route/vessel series.")
else:
    requested_month = (
        pd.Timestamp(HISTORICAL_INPUT_DATE)
        .to_period("M").to_timestamp()
    )
    if requested_month < split_date:
        raise ValueError(
            f"HISTORICAL_INPUT_DATE must be on/after the holdout cutoff {split_date:%Y-%m} "
            "so the comparison does not use a model trained on future data."
        )
    possible = selected_series[
        selected_series["date"] <= requested_month
    ].copy()

    if possible.empty:
        raise ValueError(
            "HISTORICAL_INPUT_DATE is earlier than the selected series."
        )

    historical_origin = possible.iloc[-1]

origin_date = pd.Timestamp(historical_origin["date"])
historical_future = selected_series[
    selected_series["date"] > origin_date
].head(4).copy()

if len(historical_future) < 4:
    raise ValueError(
        "The selected historical input has fewer than four subsequent "
        "observations. Choose an earlier HISTORICAL_INPUT_DATE."
    )

print("\n" + "=" * 65)
print("OUTPUT 2 — CHRONOLOGICAL HOLDOUT SAMPLE: NEXT FOUR MONTHS")
print("=" * 65)
print(f"Holdout input date: {origin_date:%Y-%m} (after training cutoff {split_date:%Y-%m})")
print(f"Route: {historical_origin['route_id']}")
print(f"Vessel type: {historical_origin['vessel_type']}")
print(f"Origin port: {historical_origin['origin_port_id']}")
print(f"Destination port: {historical_origin['destination_port_id']}")

historical_predictions = predict_four_months(
    history=selected_series,
    origin_row=historical_origin,
    first_forecast_month=historical_future.iloc[0]["date"],
    model_set=models  # trained only on dates before split_date
)

# Compare each prediction with the matching actual month.
historical_comparison = pd.DataFrame({
    "Month": historical_future["date"].to_numpy(),
    "Actual freight rate (USD/tonne)": historical_future[TARGET].to_numpy(),
    "Predicted freight rate (USD/tonne)": historical_predictions[
        "Predicted freight rate (USD/tonne)"
    ].to_numpy()
})

historical_comparison["Absolute error"] = np.abs(
    historical_comparison["Actual freight rate (USD/tonne)"]
    - historical_comparison["Predicted freight rate (USD/tonne)"]
)

display(
    historical_comparison.style.format({
        "Actual freight rate (USD/tonne)": "{:.3f}",
        "Predicted freight rate (USD/tonne)": "{:.3f}",
        "Absolute error": "{:.3f}"
    })
)

plt.figure(figsize=(10, 5))
plt.plot(
    historical_comparison["Month"],
    historical_comparison["Actual freight rate (USD/tonne)"],
    marker="o",
    label="Actual freight rate"
)
plt.plot(
    historical_comparison["Month"],
    historical_comparison["Predicted freight rate (USD/tonne)"],
    marker="s",
    linestyle="--",
    label="Predicted freight rate"
)
plt.title(
    f"Actual vs Predicted Freight Rate — Next 4 Months\n"
    f"Route: {historical_origin['route_id']} | "
    f"Vessel: {historical_origin['vessel_type']}\n"
    f"Input date: {origin_date:%Y-%m}"
)
plt.xlabel("Forecast month")
plt.ylabel("Freight rate (USD/tonne)")
plt.xticks(
    historical_comparison["Month"],
    historical_comparison["Month"].dt.strftime("%Y-%m"),
    rotation=45
)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 9. OUTPUT 3 — SYNTHETIC INPUT OUTSIDE THE DATASET
#    No graph. This uses the latest actual row as the base
#    scenario and moves the input date to the requested future date.
# ------------------------------------------------------------

synthetic_date = (
    pd.Timestamp(SYNTHETIC_INPUT_DATE)
    .to_period("M").to_timestamp()
)

if synthetic_date <= last_historical_date:
    raise ValueError(
        f"SYNTHETIC_INPUT_DATE must be after {last_historical_date:%Y-%m}. "
        "For example, set it to '2030-01-01'."
    )

latest_row = selected_series.iloc[-1].copy()

print("\n" + "=" * 65)
print("OUTPUT 3 — SYNTHETIC FUTURE INPUT")
print("=" * 65)

print("Synthetic input:")
print(f"Input date: {synthetic_date:%Y-%m}")
print(f"Route: {latest_row['route_id']}")
print(f"Vessel type: {latest_row['vessel_type']}")
print(f"Origin port: {latest_row['origin_port_id']}")
print(f"Destination port: {latest_row['destination_port_id']}")
print(
    "Market/commodity indicators: carried forward from the latest "
    "available observation; future values are not known."
)
print(
    f"Historical freight-rate information ends at "
    f"{selected_series['date'].max():%Y-%m}."
)

synthetic_origin = latest_row.copy()
synthetic_origin["date"] = synthetic_date

synthetic_forecast = predict_four_months(
    history=selected_series,
    origin_row=synthetic_origin,
    first_forecast_month=synthetic_date + pd.DateOffset(months=1),
    model_set=final_models
)

display(
    synthetic_forecast.style.format({
        "Predicted freight rate (USD/tonne)": "{:.3f}"
    })
)

print(
    "\nNote: The synthetic forecast is a model scenario, not a guarantee. "
    "It assumes the latest observed market/commodity indicators remain "
    "unchanged and uses the selected series' latest historical rate lags."
)


## Corrected next-order selection

Run this after the model-training cell above. The old cells used random market/lag values, matched orders by the wrong port pair, treated the next loaded route as empty repositioning, and added USD/tonne directly to USD costs. This section uses the vessel's post-delivery position and availability time instead.

Forecast rates are route/vessel-type estimates, not cargo-specific: the freight-rate dataset does not contain a cargo-type label. The four-month forecast window and fixed costs are assumptions. The temporal holdout metrics above describe this dataset only and do not guarantee future performance.

In [ ]:
import math
import numpy as np
import pandas as pd

# Edit these to represent the initial voyage whose returning vessel you want to assign.
INITIAL_ORIGIN_NAME = origin_port
INITIAL_DESTINATION_NAME = destination_port
INITIAL_DEPARTURE = pd.Timestamp(current_month)
INITIAL_CARGO_MT = float(quantity)
INITIAL_CARGO_TYPE = str(type_of_cargo).strip().casefold().replace(" ", "_")
REPOSITION_COST_PER_NM = 1.50
IDLE_COST_PER_DAY = 5000.00
LOAD_TIME_INCLUDED = True

# Convert names to the dataset's port IDs.
port_by_name = df2.assign(_name=df2.port_name.astype(str).str.casefold().str.strip()).set_index("_name")
def port_id_for_name(name):
    key = str(name).casefold().strip()
    if key not in port_by_name.index:
        raise ValueError(f"Unknown port name: {name}")
    row = port_by_name.loc[key]
    if isinstance(row, pd.DataFrame):
        row = row.iloc[0]
    return str(row.port_id)

initial_origin_id = port_id_for_name(INITIAL_ORIGIN_NAME)
initial_destination_id = port_id_for_name(INITIAL_DESTINATION_NAME)
initial_routes = df5[(df5.origin_port_id.astype(str) == initial_origin_id) &
                     (df5.destination_port_id.astype(str) == initial_destination_id)]
if initial_routes.empty:
    raise ValueError("The initial voyage has no route-distance record.")
initial_route = initial_routes.iloc[0]
initial_origin = df2[df2.port_id.astype(str) == initial_origin_id].iloc[0]
initial_destination = df2[df2.port_id.astype(str) == initial_destination_id].iloc[0]

# Retain vessels that can perform the submitted first voyage, including cargo, usable capacity and port dimensions.
compat = df1.cargo_compatibility.fillna("").astype(str).str.casefold().str.replace(" ", "_")
initial_vessels = df1[compat.map(lambda s: INITIAL_CARGO_TYPE in {x.strip() for x in s.split("|")})].copy()
initial_vessels = initial_vessels[
    (initial_vessels.usable_cargo_capacity_mt >= INITIAL_CARGO_MT) &
    (initial_vessels.length_overall_m <= float(initial_destination.max_vessel_loa_m)) &
    (initial_vessels.beam_m <= float(initial_destination.max_vessel_beam_m)) &
    (initial_vessels.loaded_draft_m <= float(initial_destination.max_permissible_draft_m)) &
    (initial_vessels.deadweight_tonnage_dwt <= float(initial_destination.max_berth_dwt))
].copy()

# The next order is considered only after sailing and unloading the first shipment.
initial_vessel_days = float(initial_route.distance_nm) / (24 * initial_vessels.speed_knots)
unload_days = INITIAL_CARGO_MT / max(float(initial_destination.cargo_handling_rate_tonnes_per_day), 1.0) if LOAD_TIME_INCLUDED else 0.0
initial_vessels["available_at"] = INITIAL_DEPARTURE + pd.to_timedelta(initial_vessel_days + unload_days, unit="D")
initial_vessels["current_port_id"] = initial_destination_id

# Use only the four months actually evaluated by the trained direct-horizon models.
data_cutoff = pd.Timestamp(monthly.date.max()).to_period("M").to_timestamp()
first_forecast_month = data_cutoff + pd.DateOffset(months=1)
forecast_months = [(first_forecast_month + pd.DateOffset(months=i)).to_period("M").to_timestamp() for i in range(4)]

ports_by_id = df2.drop_duplicates("port_id").set_index("port_id")
route_rows = df5.copy()
orders = df6.copy()
orders["order_time"] = pd.to_datetime(orders.order_time, errors="coerce")
orders["delivery_deadline"] = pd.to_datetime(orders.delivery_deadline, errors="coerce")
orders["quantity_mt"] = pd.to_numeric(orders.quantity_mt, errors="coerce")
orders["_status"] = orders.order_status.astype(str).str.strip().str.casefold()
orders = orders[orders._status.isin({"open", "pending", "active", "available"})].dropna(
    subset=["order_time", "delivery_deadline", "quantity_mt"]
).copy()
# Remove clearly expired / out-of-window orders up front; each candidate is still checked against its vessel's actual dates.
forecast_window_end = forecast_months[-1] + pd.offsets.MonthBegin(1)
orders = orders[(orders.delivery_deadline >= first_forecast_month) &
                (orders.order_time < forecast_window_end)].copy()

# Safely produce a forecast on the next-order route using the learned model.
# When that exact series has no history, use the latest similar vessel-class series as lag/market context,
# while replacing route IDs, distance and congestion with the requested route's values.
def predict_next_order_rate(route, vessel_type, forecast_month):
    route_id = str(route.route_id)
    origin_id = str(route.origin_port_id)
    destination_id = str(route.destination_port_id)
    exact = monthly[(monthly.route_id.astype(str) == route_id) &
                    (monthly.origin_port_id.astype(str) == origin_id) &
                    (monthly.destination_port_id.astype(str) == destination_id) &
                    (monthly.vessel_type.astype(str) == str(vessel_type))].sort_values("date")
    basis = "historical_route"
    history = exact
    if history.empty:
        analogs = monthly[monthly.vessel_type.astype(str) == str(vessel_type)].copy()
        if analogs.empty:
            return None
        latest_analogs = analogs.sort_values("date").groupby(
            GROUP_COLS, as_index=False, dropna=False
        ).tail(1).copy()
        same_origin = latest_analogs.origin_port_id.astype(str) == origin_id
        same_destination = latest_analogs.destination_port_id.astype(str) == destination_id
        if same_origin.any():
            latest_analogs = latest_analogs.loc[same_origin]
        elif same_destination.any():
            latest_analogs = latest_analogs.loc[same_destination]
        if "distance_nm" in latest_analogs.columns:
            latest_analogs["_distance_gap"] = (
                pd.to_numeric(latest_analogs.distance_nm, errors="coerce") - float(route.distance_nm)
            ).abs()
            latest_analogs = latest_analogs.sort_values("_distance_gap", na_position="last")
        analog = latest_analogs.iloc[0]
        mask = pd.Series(True, index=monthly.index)
        for col in GROUP_COLS:
            mask &= monthly[col].astype(str) == str(analog[col])
        history = monthly.loc[mask].sort_values("date")
        basis = "similar_route_model"
    if history.empty:
        return None

    last = history.iloc[-1]
    observed_rates = history[TARGET].astype(float).tolist()
    row = last.to_dict()
    row.update({
        "route_id": route_id, "origin_port_id": origin_id,
        "destination_port_id": destination_id, "vessel_type": str(vessel_type),
        "distance_nm": float(route.distance_nm),
        "origin_congestion_index": float(ports_by_id.loc[origin_id].port_congestion_index),
        "destination_congestion_index": float(ports_by_id.loc[destination_id].port_congestion_index),
    })
    for lag in [1, 2, 3, 6, 12]:
        row[f"rate_lag_{lag}"] = observed_rates[-lag] if len(observed_rates) >= lag else np.nan
    for window in [3, 6, 12]:
        row[f"rate_rolling_mean_{window}"] = float(np.mean(observed_rates[-window:])) if observed_rates else np.nan
    row["target_month_sin"] = np.sin(2 * np.pi * forecast_month.month / 12)
    row["target_month_cos"] = np.cos(2 * np.pi * forecast_month.month / 12)
    row["target_time_index"] = (forecast_month.year - BASE_DATE.year) * 12 + forecast_month.month - BASE_DATE.month
    x = pd.DataFrame([{f: row.get(f, np.nan) for f in FEATURES}])
    horizon = (forecast_month.year - first_forecast_month.year) * 12 + forecast_month.month - first_forecast_month.month + 1
    if horizon not in final_models:
        return None
    rate = max(0.0, float(final_models[horizon].predict(x[FEATURES])[0]))
    return {"rate_usd_per_tonne": rate, "forecast_basis": basis}


def cargo_tokens(value):
    return {part.strip().casefold().replace(" ", "_") for part in str(value or "").split("|") if part.strip()}

def fits_port(vessel, port):
    return (float(vessel.length_overall_m) <= float(port.max_vessel_loa_m) and
            float(vessel.beam_m) <= float(port.max_vessel_beam_m) and
            float(vessel.loaded_draft_m) <= float(port.max_permissible_draft_m) and
            float(vessel.deadweight_tonnage_dwt) <= float(port.max_berth_dwt))

candidate_plans = []
rate_cache = {}
for _, vessel in initial_vessels.iterrows():
    current_port_id = str(vessel.current_port_id)
    available_at = pd.Timestamp(vessel.available_at)
    vessel_cargo = cargo_tokens(vessel.cargo_compatibility)
    for _, order in orders.iterrows():
        origin_id, destination_id = str(order.origin_port_id), str(order.destination_port_id)
        if origin_id not in ports_by_id.index or destination_id not in ports_by_id.index:
            continue
        origin, destination = ports_by_id.loc[origin_id], ports_by_id.loc[destination_id]
        cargo = str(order.cargo_type).strip().casefold().replace(" ", "_")
        qty = float(order.quantity_mt)
        if cargo not in vessel_cargo or qty > float(vessel.usable_cargo_capacity_mt):
            continue
        if cargo not in cargo_tokens(origin.supported_cargo_types) or cargo not in cargo_tokens(destination.supported_cargo_types):
            continue
        if not fits_port(vessel, origin) or not fits_port(vessel, destination):
            continue

        if origin_id == current_port_id:
            reposition_nm = 0.0
        else:
            reposition = route_rows[(route_rows.origin_port_id.astype(str) == current_port_id) &
                                    (route_rows.destination_port_id.astype(str) == origin_id)]
            if reposition.empty:
                continue
            reposition_nm = float(reposition.iloc[0].distance_nm)
        order_route_rows = route_rows[(route_rows.origin_port_id.astype(str) == origin_id) &
                                      (route_rows.destination_port_id.astype(str) == destination_id)]
        if order_route_rows.empty:
            continue
        order_route = order_route_rows.iloc[0]
        reposition_days = reposition_nm / (24 * float(vessel.speed_knots))
        at_pickup = available_at + pd.to_timedelta(reposition_days, unit="D")
        load_days = qty / max(float(origin.cargo_handling_rate_tonnes_per_day), 1.0) if LOAD_TIME_INCLUDED else 0.0
        sailing_days = float(order_route.distance_nm) / (24 * float(vessel.speed_knots))
        for month_index, forecast_month in enumerate(forecast_months):
            month_end = forecast_month + pd.offsets.MonthBegin(1)
            load_start = max(at_pickup, pd.Timestamp(order.order_time), forecast_month)
            if load_start >= month_end:
                continue
            delivery_arrival = load_start + pd.to_timedelta(load_days + sailing_days, unit="D")
            if delivery_arrival > pd.Timestamp(order.delivery_deadline):
                continue
            rate_key = (str(order_route.route_id), origin_id, destination_id,
                        str(vessel.vessel_type), forecast_month.strftime("%Y-%m"))
            if rate_key not in rate_cache:
                rate_cache[rate_key] = predict_next_order_rate(
                    order_route, vessel.vessel_type, forecast_month
                )
            prediction = rate_cache[rate_key]
            if prediction is None:
                continue
            rate = prediction["rate_usd_per_tonne"]
            freight_cost = rate * qty
            idle_days = int(math.ceil(max(0.0, (load_start - at_pickup).total_seconds() / 86400)))
            reposition_cost = reposition_nm * REPOSITION_COST_PER_NM
            idle_cost = idle_days * IDLE_COST_PER_DAY
            total_cost = freight_cost + reposition_cost + idle_cost
            candidate_plans.append({
                "vessel_id": str(vessel.vessel_id), "vessel_name": str(vessel.vessel_name),
                "order_id": str(order.order_id), "cargo_type": str(order.cargo_type),
                "quantity_mt": qty, "origin_port_id": origin_id, "origin_port": str(origin.port_name),
                "destination_port_id": destination_id, "destination_port": str(destination.port_name),
                "forecast_month": forecast_month.strftime("%Y-%m"),
                "vessel_available_at": available_at.date().isoformat(),
                "loading_start": load_start.date().isoformat(),
                "estimated_delivery": delivery_arrival.date().isoformat(),
                "delivery_deadline": pd.Timestamp(order.delivery_deadline).date().isoformat(),
                "forecast_rate_usd_per_tonne": round(rate, 3),
                "forecast_basis": prediction["forecast_basis"],
                "freight_cost_usd": round(freight_cost, 2),
                "reposition_distance_nm": round(reposition_nm, 1),
                "reposition_cost_usd": round(reposition_cost, 2),
                "reposition_transit_days": round(reposition_days, 2),
                "idle_days": idle_days, "idle_cost_usd": round(idle_cost, 2),
                "additional_cost_usd": round(reposition_cost + idle_cost, 2),
                "effective_rate_usd_per_tonne": round(total_cost / qty, 3),
                "total_order_cost_usd": round(total_cost, 2),
            })

if not candidate_plans:
    print("No eligible next order was found in the four-month forecast window.")
    print("Check active order status, route coverage, cargo/capacity, and deadline.")
else:
    next_order_results = pd.DataFrame(candidate_plans).sort_values(
        ["effective_rate_usd_per_tonne", "total_order_cost_usd", "delivery_deadline", "order_id"]
    ).reset_index(drop=True)
    print("Next-order cost = forecast rate × order quantity + repositioning + idle cost.")
    print("Orders are ranked by all-in rate: forecast rate + (repositioning + idle costs) / order quantity.")
    display(next_order_results.head(20))

    # Rank by the all-in rate per tonne, so large and small orders are comparable.
    best_per_ship = next_order_results.loc[
        next_order_results.groupby("vessel_id")["effective_rate_usd_per_tonne"].idxmin()
    ].sort_values(["effective_rate_usd_per_tonne", "total_order_cost_usd"]).reset_index(drop=True)
    print("Best next order for each vessel, ranked by all-in USD/tonne:")
    display(best_per_ship)
    print("Lowest all-in-rate vessel/order plan:")
    display(best_per_ship.head(1))

print(f"Data cutoff: {data_cutoff:%Y-%m}. Forecast months: {[m.strftime('%Y-%m') for m in forecast_months]}.")
print(f"Fixed assumptions: ${REPOSITION_COST_PER_NM:,.2f}/nm repositioning, ${IDLE_COST_PER_DAY:,.2f}/idle day.")
